# Full 57k joint MPRA library — koo-model DeepLIFT preds + attributions

The koo standardized AlphaGenome encoder finetuned models already scored the full
joint LentiMPRA library (56,980 rows). We **do not re-score** here (no GPU in this
kernel) — we just **load** the existing normal-DeepLIFT artifact and organize it
into the MoConsSwap data dir.

Source: `Hippo_dependency_mpra/genomic_targets/data/deeplift_attributions_standardtorch_convfix.npz`
(latest "convfix" standardized-torch koo run, 2026-05-11), aligned to the original
56,980-row CSV order:
- `predictions_{HepG2,K562,WTC11}` — (56980,) scalar preds
- `attr_{HepG2,K562,WTC11}` — (56980, 4, 200) DeepLIFT attributions

In [13]:
from pathlib import Path
import glob, re
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

LIB_CSV = Path(
    "/grid/koo/home/pmantill/projects/Virtual_Experiments/Hippo_axis/"
    "Hippo_dependency_mpra/data/joint_library_combined.csv"
)
# Existing koo standardized-torch attribution artifacts (aligned to the
# original 56,980-row CSV order). We LOAD these — no re-scoring.
GT_DATA = Path(
    "/grid/koo/home/pmantill/projects/Virtual_Experiments/Hippo_axis/"
    "Hippo_dependency_mpra/genomic_targets/data"
)
DEEPLIFT_NPZ = GT_DATA / "deeplift_attributions_standardtorch_convfix.npz"
IG100_SHARD_DIR = GT_DATA / "grad_shards_standardtorch_ig100"  # IG, 100 dinuc shuffles

CELL_TYPES = ["HepG2", "K562", "WTC11"]

# Organized output dir for MoConsSwap.
KOO_DIR = Path(
    "/grid/koo/home/pmantill/projects/Virtual_Experiments/MoConsSwap_mpra/"
    "data/koo_attributions"
)
KOO_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
lib = pd.read_csv(LIB_CSV)
print(lib.shape)              # (56980, 10)
# 2 of the 56980 rows have a NaN sequence; the rest are 230 bp inserts
lib = lib[lib["sequence"].notna()].reset_index(drop=True)
print("after dropping NaN sequence:", lib.shape)  # (56978, 10)
assert (lib["sequence"].str.len() == 230).all(), "expected 230 bp inserts"
lib.head()

(56980, 10)
after dropping NaN sequence: (56978, 10)


,name,category,chr_hg38,start_hg38,stop_hg38,str_hg38,HepG2_log2FC,K562_log2FC,WTC11_log2FC,sequence
0,WTC11_seq9987_F,"putative enhancer, WTC11",10,88524112.0,88524312.0,+,0.320,-0.439,-1.505,AGGACCGGATCAACTCTGATTATATTAAAAAAAAAAGCTCTTTAGG...
1,WTC11_seq998_F,"putative enhancer, WTC11",1,19445991.0,19446191.0,+,0.003,-0.217,-0.462,AGGACCGGATCAACTAACAGAGGAAGCTGGAGGCCTCTCGGCATCA...
2,WTC11_seq9970_F,"putative enhancer, WTC11",10,87675767.0,87675967.0,+,-0.794,-1.026,-1.414,AGGACCGGATCAACTAATATGATTGAATGGATCTGTGTTGGGGAAT...
3,WTC11_seq9967_F,"putative enhancer, WTC11",10,87622705.0,87622905.0,+,-0.485,-0.592,-1.346,AGGACCGGATCAACTCTCAGTAGTTCCACAGAAGTCCCAGGATTGC...
4,WTC11_seq996_F,"putative enhancer, WTC11",1,19410480.0,19410680.0,+,-1.240,-1.036,-1.344,AGGACCGGATCAACTTATGGGTGGGGGGTGGTGGGGGGCTGCAATA...


In [15]:
# Load existing koo attributions (NO re-scoring). Both artifacts are in the
# original 56,980-row CSV order; we apply the same notna-sequence mask used in
# cell-2 so everything stays row-aligned with `lib` (56,978 rows).
raw_seq = pd.read_csv(LIB_CSV, usecols=["sequence"])["sequence"]
keep = raw_seq.notna().values                       # (56980,) -> 56978 True
assert keep.sum() == len(lib), (keep.sum(), len(lib))

# --- DeepLIFT (convfix standardized-torch) ---
dl = np.load(DEEPLIFT_NPZ, allow_pickle=True)
preds = {ct: dl[f"predictions_{ct}"][keep].astype(np.float32) for ct in CELL_TYPES}
deeplift = {ct: dl[f"attr_{ct}"][keep].astype(np.float32) for ct in CELL_TYPES}

# --- Integrated gradients, 100 dinuc shuffles (assemble 33 ordered shards/ct) ---
ig100 = {}
for ct in CELL_TYPES:
    shards = sorted(
        glob.glob(str(IG100_SHARD_DIR / f"{ct}_shard_*.npz")),
        key=lambda s: int(re.search(r"_shard_(\d+)\.npz$", s).group(1)),
    )
    arr = np.concatenate([np.load(s)["intgrad"] for s in shards], axis=0)
    assert arr.shape[0] == keep.shape[0], (ct, arr.shape)
    ig100[ct] = arr[keep].astype(np.float32)

print("rows:", len(lib))
for ct in CELL_TYPES:
    print(f"  {ct}: pred {preds[ct].shape}  deeplift {deeplift[ct].shape}  "
          f"ig100 {ig100[ct].shape}")

rows: 56978
  HepG2: pred (56978,)  deeplift (56978, 4, 200)  ig100 (56978, 4, 200)
  K562: pred (56978,)  deeplift (56978, 4, 200)  ig100 (56978, 4, 200)
  WTC11: pred (56978,)  deeplift (56978, 4, 200)  ig100 (56978, 4, 200)


In [16]:
# Attach koo preds to the library, sanity-check vs measured log2FC, then write
# the organized MoConsSwap artifacts.
out = lib.copy()
for ct in CELL_TYPES:
    out[f"{ct}_pred_koo"] = preds[ct]
    m = out[f"{ct}_log2FC"].notna()
    r, _ = pearsonr(out.loc[m, f"{ct}_log2FC"], out.loc[m, f"{ct}_pred_koo"])
    print(f"{ct}: pearson(measured log2FC, koo pred) = {r:.4f}  (n={m.sum()})")

# CSV (not parquet) — the Hippo_agft_venv kernel has no pyarrow/fastparquet.
preds_path = KOO_DIR / "joint_library_combined_koo_preds.csv"
out.to_csv(preds_path, index=False)

# Attribution arrays (56,978 x 4 x 200), row-aligned to the preds table above.
dl_path = KOO_DIR / "koo_deeplift_attributions.npz"
ig_path = KOO_DIR / "koo_ig100_attributions.npz"
np.savez_compressed(dl_path, **{f"attr_{ct}": deeplift[ct] for ct in CELL_TYPES})
np.savez_compressed(ig_path, **{f"intgrad_{ct}": ig100[ct] for ct in CELL_TYPES})

for p in (preds_path, dl_path, ig_path):
    print("wrote", p)

HepG2: pearson(measured log2FC, koo pred) = 0.8906  (n=56978)
K562: pearson(measured log2FC, koo pred) = 0.8979  (n=56975)
WTC11: pearson(measured log2FC, koo pred) = 0.8498  (n=56975)
wrote /grid/koo/home/pmantill/projects/Virtual_Experiments/MoConsSwap_mpra/data/koo_attributions/joint_library_combined_koo_preds.csv
wrote /grid/koo/home/pmantill/projects/Virtual_Experiments/MoConsSwap_mpra/data/koo_attributions/koo_deeplift_attributions.npz
wrote /grid/koo/home/pmantill/projects/Virtual_Experiments/MoConsSwap_mpra/data/koo_attributions/koo_ig100_attributions.npz
